In [ ]:
import gc
import json
import os

import matplotlib.pyplot as plt
import monai
import numpy as np
import pandas as pd
import torch
from monai.data import Dataset, list_data_collate
from monai.inferers import sliding_window_inference
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric, HausdorffDistanceMetric, SurfaceDistanceMetric
from monai.networks.nets import UNet
from monai.transforms import (
    CropForegroundd,
    DivisiblePadd,
    EnsureChannelFirstd,
    Lambdad,
    LoadImaged,
    NormalizeIntensityd,
    Orientationd,
    RandCropByPosNegLabeld,
    RandFlipd,
    RandGaussianNoised,
    RandRotate90d,
    Spacingd,
    SpatialPadd,
    ToTensord,
)
from monai.utils import set_determinism
from sklearn.model_selection import KFold, train_test_split
from torch.utils.data import DataLoader


# ============================================================
# CONFIGURACION GENERAL
# ============================================================

DATA_ROOT = "${HOME}/TESIS/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"
OUT_DIR = "${HOME}/TESIS/TESIS_RESULTS_KFOLD"

# K-fold externo: estos folds se usan para la evaluacion final.
K_FOLDS = 5

# Validacion interna dentro del train de cada fold.
# Esta validacion interna es la unica que se usa para elegir el mejor epoch.
INNER_VAL_FRACTION = 0.15

NUM_EPOCHS = 20
BATCH_SIZE = 4
VAL_INTERVAL = 1
LR = 1e-4

ROI_SIZE = (128, 128, 128)
SW_BATCH_SIZE = 1
OVERLAP = 0.5
THRESHOLD = 0.5

NUM_WORKERS_TRAIN = 16
NUM_WORKERS_EVAL = 16
SEED = 0

# Para depurar rapido. En el experimento final dejar en False.
PROBAR_CON_POCOS_DATOS = False
CANTIDAD_PACIENTES = 20

# Para probar un solo fold se puede usar, por ejemplo: FOLDS_TO_RUN = [1]
# En el experimento final dejar None para correr todos los folds.
FOLDS_TO_RUN = None

# Si un fold ya tiene marcador de finalizacion, se salta al volver a ejecutar.
# Esto permite continuar desde el siguiente fold si el proceso se detiene.
RESUME_COMPLETED_FOLDS = True

# Si necesitas repetir un fold aunque ya este completado, ponlo aqui.
# Ejemplo: FORCE_RERUN_FOLDS = [2]
FORCE_RERUN_FOLDS = []

os.makedirs(OUT_DIR, exist_ok=True)
set_determinism(seed=SEED)


# ============================================================
# DATASET Y TRANSFORMACIONES
# ============================================================

def build_datalist(root):
    """Genera la lista de casos BraTS con rutas y case_id."""
    datalist = []

    if not os.path.exists(root):
        print(f"DATA_ROOT no existe: {root}")
        return datalist

    for case in sorted(os.listdir(root)):
        case_dir = os.path.join(root, case)
        if not os.path.isdir(case_dir):
            continue

        files = os.listdir(case_dir)
        pref = None

        for filename in files:
            if filename.endswith("-t1n.nii.gz"):
                pref = filename.replace("-t1n.nii.gz", "")
                break

        if pref is None:
            continue

        needed = [
            f"{pref}-t1n.nii.gz",
            f"{pref}-t1c.nii.gz",
            f"{pref}-t2w.nii.gz",
            f"{pref}-t2f.nii.gz",
            f"{pref}-seg.nii.gz",
        ]

        if not all(filename in files for filename in needed):
            continue

        datalist.append(
            {
                "image": [
                    os.path.join(case_dir, f"{pref}-t1n.nii.gz"),
                    os.path.join(case_dir, f"{pref}-t1c.nii.gz"),
                    os.path.join(case_dir, f"{pref}-t2w.nii.gz"),
                    os.path.join(case_dir, f"{pref}-t2f.nii.gz"),
                ],
                "label": os.path.join(case_dir, f"{pref}-seg.nii.gz"),
                "case_id": case,
            }
        )

    return datalist


def get_transforms():
    """Transformaciones para entrenamiento y evaluacion."""

    def binarize_label(x):
        return (x > 0).float()

    train_transforms = monai.transforms.Compose(
        [
            LoadImaged(keys=["image", "label"]),
            EnsureChannelFirstd(keys=["image", "label"]),
            Spacingd(
                keys=["image", "label"],
                pixdim=(1.0, 1.0, 1.0),
                mode=("bilinear", "nearest"),
            ),
            Orientationd(keys=["image", "label"], axcodes="RAS"),
            Lambdad(keys=["label"], func=binarize_label),
            NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),
            CropForegroundd(keys=["image", "label"], source_key="image"),
            SpatialPadd(keys=["image", "label"], spatial_size=ROI_SIZE),
            RandCropByPosNegLabeld(
                keys=["image", "label"],
                label_key="label",
                spatial_size=ROI_SIZE,
                pos=1,
                neg=1,
                num_samples=2,
            ),
            RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
            RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
            RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
            RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
            RandGaussianNoised(keys=["image"], prob=0.15, std=0.1),
            ToTensord(keys=["image", "label"]),
        ]
    )

    eval_transforms = monai.transforms.Compose(
        [
            LoadImaged(keys=["image", "label"]),
            EnsureChannelFirstd(keys=["image", "label"]),
            Spacingd(
                keys=["image", "label"],
                pixdim=(1.0, 1.0, 1.0),
                mode=("bilinear", "nearest"),
            ),
            Orientationd(keys=["image", "label"], axcodes="RAS"),
            Lambdad(keys=["label"], func=binarize_label),
            NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),
            CropForegroundd(keys=["image", "label"], source_key="image"),
            DivisiblePadd(keys=["image", "label"], k=16),
            ToTensord(keys=["image", "label"]),
        ]
    )

    return train_transforms, eval_transforms


# ============================================================
# MODELO Y UTILIDADES
# ============================================================

def create_model(device):
    """Crea la misma UNet 3D usada en el entrenamiento original."""
    return UNet(
        spatial_dims=3,
        in_channels=4,
        out_channels=1,
        channels=(16, 32, 64, 128, 256),
        strides=(2, 2, 2, 2),
        num_res_units=2,
        norm="batch",
    ).to(device)


def load_model_weights(model, model_path, device):
    """Carga pesos de un checkpoint .pth."""
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"No existe el checkpoint: {model_path}")

    try:
        state_dict = torch.load(model_path, map_location=device, weights_only=True)
    except TypeError:
        state_dict = torch.load(model_path, map_location=device)

    model.load_state_dict(state_dict)
    model.eval()
    return model


def get_case_id(batch_data, index):
    """Extrae case_id desde el batch de MONAI/DataLoader."""
    case_id = batch_data.get("case_id", f"case_{index:04d}")

    if isinstance(case_id, (list, tuple)):
        return str(case_id[0])

    return str(case_id)


def make_inner_split(train_pool_files, fold):
    """Separa train real y validacion interna para seleccionar checkpoint."""
    if len(train_pool_files) < 2:
        raise ValueError("No hay suficientes casos para crear validacion interna.")

    inner_val_count = max(1, int(round(len(train_pool_files) * INNER_VAL_FRACTION)))
    inner_val_count = min(inner_val_count, len(train_pool_files) - 1)

    train_files, inner_val_files = train_test_split(
        train_pool_files,
        test_size=inner_val_count,
        shuffle=True,
        random_state=SEED + fold,
    )

    return train_files, inner_val_files


def infer_binary_mask(model, inputs):
    """Inferencia con sliding window y umbral binario."""
    logits = sliding_window_inference(
        inputs=inputs,
        roi_size=ROI_SIZE,
        sw_batch_size=SW_BATCH_SIZE,
        predictor=model,
        overlap=OVERLAP,
    )
    probs = torch.sigmoid(logits)
    return (probs > THRESHOLD).float()


def clean_cuda():
    """Libera memoria entre folds."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def get_fold_paths(fold):
    """Rutas estandar de archivos generados por cada fold."""
    fold_out_dir = os.path.join(OUT_DIR, f"fold_{fold:02d}")
    return {
        "fold_out_dir": fold_out_dir,
        "checkpoint": os.path.join(fold_out_dir, f"best_metric_model_fold_{fold:02d}.pth"),
        "training_log": os.path.join(fold_out_dir, f"training_log_fold_{fold:02d}.csv"),
        "heldout_results": os.path.join(fold_out_dir, f"heldout_results_fold_{fold:02d}.csv"),
        "completion_marker": os.path.join(fold_out_dir, "FOLD_COMPLETED.json"),
    }


def read_json(path):
    """Lee un JSON simple."""
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def write_json(path, data):
    """Guarda un JSON simple."""
    with open(path, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=2)


def is_fold_completed(fold):
    """Determina si un fold termino correctamente en una ejecucion anterior."""
    paths = get_fold_paths(fold)
    marker_path = paths["completion_marker"]

    if not os.path.exists(marker_path):
        return False, None

    try:
        marker = read_json(marker_path)
    except Exception as exc:
        print(f"Marcador corrupto en fold {fold}: {marker_path}. Se repetira el fold. Error: {exc}")
        return False, None

    required_paths = [
        marker.get("checkpoint", paths["checkpoint"]),
        marker.get("training_log", paths["training_log"]),
        marker.get("heldout_results_csv", paths["heldout_results"]),
    ]

    missing_paths = [path for path in required_paths if not path or not os.path.exists(path)]

    if missing_paths:
        print(f"Fold {fold} tenia marcador, pero faltan archivos. Se repetira el fold.")
        for path in missing_paths:
            print("  Falta:", path)
        return False, marker

    return bool(marker.get("completed", False)), marker


def load_completed_fold_outputs(fold, marker):
    """Carga resultados/logs de un fold ya completado para resumenes globales."""
    paths = get_fold_paths(fold)
    results_csv = marker.get("heldout_results_csv", paths["heldout_results"])
    log_path = marker.get("training_log", paths["training_log"])

    fold_results_df = pd.read_csv(results_csv)
    training_log_df = pd.read_csv(log_path)

    run_summary_row = {
        "Fold": fold,
        "train_pool_n": marker.get("train_pool_n", np.nan),
        "train_n": marker.get("train_n", np.nan),
        "inner_val_n": marker.get("inner_val_n", np.nan),
        "held_out_n": marker.get("held_out_n", np.nan),
        "best_inner_val_dice": marker.get("best_inner_val_dice", np.nan),
        "best_epoch": marker.get("best_epoch", np.nan),
        "checkpoint": marker.get("checkpoint", paths["checkpoint"]),
        "resumed_from_completed_fold": True,
    }

    return fold_results_df, training_log_df, run_summary_row


def write_fold_completion_marker(
    fold,
    train_pool_files,
    train_files,
    inner_val_files,
    held_out_files,
    best_model_path,
    log_path,
    fold_results_csv,
    best_inner_dice,
    best_epoch,
):
    """Marca un fold como completado solo despues de entrenar, evaluar y graficar."""
    paths = get_fold_paths(fold)
    marker = {
        "completed": True,
        "fold": fold,
        "train_pool_n": len(train_pool_files),
        "train_n": len(train_files),
        "inner_val_n": len(inner_val_files),
        "held_out_n": len(held_out_files),
        "best_inner_val_dice": float(best_inner_dice) if np.isfinite(best_inner_dice) else None,
        "best_epoch": int(best_epoch),
        "checkpoint": best_model_path,
        "training_log": log_path,
        "heldout_results_csv": fold_results_csv,
    }
    write_json(paths["completion_marker"], marker)
    print("Marcador de fold completado:", paths["completion_marker"])


# ============================================================
# ENTRENAMIENTO POR FOLD
# ============================================================

def compute_inner_validation_dice(model, val_loader, device):
    """Calcula Dice en la validacion interna de seleccion de modelo."""
    dice_metric = DiceMetric(include_background=True, reduction="mean")

    model.eval()
    dice_metric.reset()

    with torch.no_grad():
        for val_data in val_loader:
            val_inputs = val_data["image"].to(device)
            val_labels = val_data["label"].to(device)
            val_outputs = infer_binary_mask(model, val_inputs)
            dice_metric(y_pred=val_outputs, y=val_labels)

    metric = dice_metric.aggregate().item()
    dice_metric.reset()
    return float(metric)


def train_one_fold(fold, train_files, inner_val_files, train_transforms, eval_transforms, device):
    """Entrena un fold y guarda el mejor checkpoint por validacion interna."""
    fold_out_dir = os.path.join(OUT_DIR, f"fold_{fold:02d}")
    os.makedirs(fold_out_dir, exist_ok=True)

    train_ds = Dataset(data=train_files, transform=train_transforms)
    inner_val_ds = Dataset(data=inner_val_files, transform=eval_transforms)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS_TRAIN,
        pin_memory=torch.cuda.is_available(),
        collate_fn=list_data_collate,
    )

    inner_val_loader = DataLoader(
        inner_val_ds,
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS_EVAL,
        pin_memory=torch.cuda.is_available(),
    )

    model = create_model(device)

    loss_function = DiceCELoss(
        include_background=True,
        sigmoid=True,
        to_onehot_y=False,
        squared_pred=True,
        smooth_nr=1e-5,
        smooth_dr=1e-5,
        lambda_dice=0.7,
        lambda_ce=0.3,
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    best_metric = -1.0
    best_metric_epoch = -1
    best_model_path = os.path.join(fold_out_dir, f"best_metric_model_fold_{fold:02d}.pth")
    log_path = os.path.join(fold_out_dir, f"training_log_fold_{fold:02d}.csv")
    training_log = []

    print("\n" + "=" * 70)
    print(f"FOLD {fold}/{K_FOLDS}")
    print(f"Train real: {len(train_files)} | Validacion interna: {len(inner_val_files)}")
    print("=" * 70)

    for epoch in range(1, NUM_EPOCHS + 1):
        print("-" * 30)
        print(f"Fold {fold} - Epoch {epoch}/{NUM_EPOCHS}")
        model.train()
        epoch_loss = 0.0
        step = 0

        for batch_data in train_loader:
            step += 1
            inputs = batch_data["image"].to(device)
            labels = batch_data["label"].to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = loss_function(outputs, labels)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            if step % 10 == 0:
                print(f"   step {step}/{len(train_loader)}, loss: {loss.item():.4f}", end="\r")

        epoch_loss /= max(step, 1)
        print(f"\n  Train loss: {epoch_loss:.4f}")

        inner_val_dice = np.nan

        if epoch % VAL_INTERVAL == 0:
            inner_val_dice = compute_inner_validation_dice(model, inner_val_loader, device)
            print(f"  Inner Val Dice: {inner_val_dice:.4f}")

            if np.isfinite(inner_val_dice) and inner_val_dice > best_metric:
                best_metric = inner_val_dice
                best_metric_epoch = epoch
                torch.save(model.state_dict(), best_model_path)
                print(f"  >>> Nuevo mejor checkpoint interno guardado en epoch {epoch}")

        training_log.append(
            {
                "fold": fold,
                "epoch": epoch,
                "train_loss": epoch_loss,
                "inner_val_dice": inner_val_dice,
                "best_inner_val_dice_so_far": best_metric if best_metric >= 0 else np.nan,
            }
        )
        pd.DataFrame(training_log).to_csv(log_path, index=False)

    if best_metric_epoch == -1:
        best_metric_epoch = NUM_EPOCHS
        best_metric = np.nan
        torch.save(model.state_dict(), best_model_path)
        print("No se obtuvo Dice interno valido; se guardo el ultimo modelo.")

    print(
        f"\nFold {fold} terminado. Mejor Inner Val Dice: {best_metric:.4f} "
        f"en epoch {best_metric_epoch}"
    )
    print("Checkpoint:", best_model_path)
    print("Log:", log_path)

    del model
    clean_cuda()

    return best_model_path, log_path, best_metric, best_metric_epoch


# ============================================================
# EVALUACION HELD-OUT POR FOLD
# ============================================================

def compute_binary_metrics(pred_mask, labels):
    """Metricas voxel-wise para una mascara binaria."""
    pred_bool = pred_mask.bool()
    label_bool = labels.bool()

    tp = torch.logical_and(pred_bool, label_bool).sum().item()
    fp = torch.logical_and(pred_bool, torch.logical_not(label_bool)).sum().item()
    tn = torch.logical_and(torch.logical_not(pred_bool), torch.logical_not(label_bool)).sum().item()
    fn = torch.logical_and(torch.logical_not(pred_bool), label_bool).sum().item()

    dice_den = 2.0 * tp + fp + fn
    iou_den = tp + fp + fn
    sens_den = tp + fn
    spec_den = tn + fp
    prec_den = tp + fp
    acc_den = tp + tn + fp + fn

    dice = 1.0 if dice_den == 0 else (2.0 * tp) / dice_den
    iou = 1.0 if iou_den == 0 else tp / iou_den
    sens = 1.0 if sens_den == 0 else tp / sens_den
    spec = 1.0 if spec_den == 0 else tn / spec_den
    prec = 1.0 if prec_den == 0 else tp / prec_den
    acc = np.nan if acc_den == 0 else (tp + tn) / acc_den
    b_acc = (sens + spec) / 2.0

    return {
        "Dice": dice,
        "IoU": iou,
        "Recall_Sens": sens,
        "Specificity": spec,
        "Precision": prec,
        "Accuracy": acc,
        "Balanced_Acc": b_acc,
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn,
    }


def evaluate_fold_checkpoint(fold, model_path, held_out_files, eval_transforms, device):
    """Evalua el checkpoint elegido en el fold held-out no usado para seleccionar epoch."""
    fold_out_dir = os.path.join(OUT_DIR, f"fold_{fold:02d}")
    os.makedirs(fold_out_dir, exist_ok=True)

    eval_ds = Dataset(data=held_out_files, transform=eval_transforms)
    eval_loader = DataLoader(
        eval_ds,
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS_EVAL,
        pin_memory=torch.cuda.is_available(),
    )

    model = create_model(device)
    model = load_model_weights(model, model_path, device)
    model.eval()

    hd95_metric = HausdorffDistanceMetric(
        include_background=True,
        percentile=95,
        reduction="none",
    )
    asd_metric = SurfaceDistanceMetric(
        include_background=True,
        symmetric=True,
        reduction="none",
    )

    detailed_results = []

    print(f"\nEvaluacion held-out del fold {fold}: {len(held_out_files)} casos")

    with torch.no_grad():
        for i, val_data in enumerate(eval_loader):
            inputs = val_data["image"].to(device)
            labels = val_data["label"].to(device)
            case_id = get_case_id(val_data, i)

            print(f"Fold {fold} - procesando {i + 1}/{len(eval_loader)}: {case_id}", end="\r")

            pred_mask = infer_binary_mask(model, inputs)
            voxel_metrics = compute_binary_metrics(pred_mask, labels)

            hd95_val = np.nan
            asd_val = np.nan

            if pred_mask.sum() > 0 and labels.sum() > 0:
                try:
                    hd95_metric.reset()
                    hd95_metric(y_pred=pred_mask, y=labels)
                    hd95_val = float(hd95_metric.aggregate().cpu().numpy().reshape(-1)[0])
                except Exception:
                    hd95_val = np.nan

                try:
                    asd_metric.reset()
                    asd_metric(y_pred=pred_mask, y=labels)
                    asd_val = float(asd_metric.aggregate().cpu().numpy().reshape(-1)[0])
                except Exception:
                    asd_val = np.nan

            voxel_vol_ml = 1.0 * 1.0 * 1.0 / 1000.0
            vol_gt = labels.sum().item() * voxel_vol_ml
            vol_pred = pred_mask.sum().item() * voxel_vol_ml

            row = {
                "Fold": fold,
                "Case_ID": case_id,
                **voxel_metrics,
                "HD95": hd95_val,
                "ASD": asd_val,
                "Vol_GT_mL": vol_gt,
                "Vol_Pred_mL": vol_pred,
                "Vol_Error_mL": vol_pred - vol_gt,
                "Model_Path": model_path,
            }
            detailed_results.append(row)

    print("\nEvaluacion del fold terminada.")

    df = pd.DataFrame(detailed_results)
    csv_path = os.path.join(fold_out_dir, f"heldout_results_fold_{fold:02d}.csv")
    df.to_csv(csv_path, index=False)
    print("CSV fold:", csv_path)

    del model
    clean_cuda()

    return df, csv_path


# ============================================================
# RESUMENES Y GRAFICAS
# ============================================================

def save_numeric_summary(df, out_dir, filename):
    """Guarda mean/std/median/min/max de las columnas numericas."""
    numeric_df = df.select_dtypes(include=[np.number]).copy()
    numeric_df = numeric_df.drop(columns=["Fold"], errors="ignore")

    summary = pd.concat(
        [
            numeric_df.mean(numeric_only=True).to_frame(name="Mean").T,
            numeric_df.std(numeric_only=True).to_frame(name="Std").T,
            numeric_df.median(numeric_only=True).to_frame(name="Median").T,
            numeric_df.min(numeric_only=True).to_frame(name="Min").T,
            numeric_df.max(numeric_only=True).to_frame(name="Max").T,
        ],
        axis=0,
    )

    path = os.path.join(out_dir, filename)
    summary.to_csv(path, index=True)
    print("Resumen:", path)
    return summary, path


def generate_training_curves(log_path, out_dir, fold_label=None):
    """Genera curvas de loss e Inner Val Dice desde un log de entrenamiento."""
    if not os.path.exists(log_path):
        print(f"No existe log: {log_path}")
        return

    df = pd.read_csv(log_path)
    required = {"epoch", "train_loss", "inner_val_dice"}
    missing = required.difference(df.columns)

    if missing:
        print(f"Log sin columnas requeridas {missing}: {log_path}")
        return

    df["epoch"] = pd.to_numeric(df["epoch"], errors="coerce")
    df["train_loss"] = pd.to_numeric(df["train_loss"], errors="coerce")
    df["inner_val_dice"] = pd.to_numeric(df["inner_val_dice"], errors="coerce")
    df = df.dropna(subset=["epoch"]).sort_values("epoch")

    if df.empty:
        print(f"Log vacio: {log_path}")
        return

    title_suffix = f" - Fold {fold_label}" if fold_label is not None else ""
    min_epoch = int(df["epoch"].min())
    max_epoch = int(df["epoch"].max())

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(df["epoch"], df["train_loss"], marker="o", markersize=3, linewidth=2)
    axes[0].set_title(f"Training Loss{title_suffix}", fontsize=13, fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, linestyle="--", alpha=0.5)

    valid_dice = df.dropna(subset=["inner_val_dice"])
    axes[1].plot(df["epoch"], df["inner_val_dice"], marker="o", markersize=3, linewidth=2)

    if not valid_dice.empty:
        best_idx = valid_dice["inner_val_dice"].idxmax()
        best_epoch = int(valid_dice.loc[best_idx, "epoch"])
        best_dice = float(valid_dice.loc[best_idx, "inner_val_dice"])
        axes[1].scatter(best_epoch, best_dice, s=80, zorder=5)
        axes[1].annotate(
            f"Best = {best_dice:.4f}\nEpoch {best_epoch}",
            xy=(best_epoch, best_dice),
            xytext=(best_epoch, max(0.02, best_dice - 0.08)),
            arrowprops=dict(arrowstyle="->"),
            fontsize=10,
            ha="center",
        )

    axes[1].set_title(f"Inner Validation Dice{title_suffix}", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Dice")
    axes[1].set_ylim(0, 1.02)
    axes[1].grid(True, linestyle="--", alpha=0.5)

    tick_step = 5 if max_epoch <= 100 else 10
    for ax in axes:
        ax.set_xlim(min_epoch, max_epoch)
        ax.set_xticks(np.arange(min_epoch, max_epoch + 1, tick_step))

    plt.tight_layout()

    suffix = f"_fold_{int(fold_label):02d}" if fold_label is not None else ""
    png_path = os.path.join(out_dir, f"training_curves{suffix}.png")
    pdf_path = os.path.join(out_dir, f"training_curves{suffix}.pdf")

    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)

    print("Grafica entrenamiento PNG:", png_path)
    print("Grafica entrenamiento PDF:", pdf_path)


def generate_kfold_training_overview(all_logs_csv, out_dir):
    """Grafica curvas comparativas de todos los folds."""
    if not os.path.exists(all_logs_csv):
        return

    df = pd.read_csv(all_logs_csv)
    if df.empty or "fold" not in df.columns:
        return

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    for fold, fold_df in df.groupby("fold"):
        fold_df = fold_df.sort_values("epoch")
        axes[0].plot(fold_df["epoch"], fold_df["train_loss"], linewidth=1.8, label=f"Fold {fold}")
        axes[1].plot(fold_df["epoch"], fold_df["inner_val_dice"], linewidth=1.8, label=f"Fold {fold}")

    axes[0].set_title("Training Loss by Fold", fontsize=13, fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, linestyle="--", alpha=0.5)
    axes[0].legend()

    axes[1].set_title("Inner Validation Dice by Fold", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Dice")
    axes[1].set_ylim(0, 1.02)
    axes[1].grid(True, linestyle="--", alpha=0.5)
    axes[1].legend()

    plt.tight_layout()

    png_path = os.path.join(out_dir, "kfold_training_overview.png")
    pdf_path = os.path.join(out_dir, "kfold_training_overview.pdf")
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)

    print("Grafica k-fold entrenamiento:", png_path)


def generate_thesis_plots(results_csv_path, out_dir, prefix="kfold"):
    """Genera las graficas finales de evaluacion para tesis."""
    if not os.path.exists(results_csv_path):
        print(f"No existe CSV de resultados: {results_csv_path}")
        return

    df_res = pd.read_csv(results_csv_path)

    metrics_to_plot = [
        "Dice",
        "IoU",
        "Recall_Sens",
        "Specificity",
        "Precision",
        "Balanced_Acc",
    ]
    metrics_to_plot = [metric for metric in metrics_to_plot if metric in df_res.columns]

    if metrics_to_plot:
        plt.figure(figsize=(11, 6))
        df_res.boxplot(column=metrics_to_plot)
        plt.title("Distribution of Segmentation Metrics", fontsize=13, fontweight="bold")
        plt.ylabel("Score")
        plt.ylim(0, 1.05)
        plt.grid(axis="y", linestyle="--", alpha=0.5)
        plt.xticks(rotation=25)
        plt.tight_layout()
        save_path = os.path.join(out_dir, f"{prefix}_boxplot_metrics.png")
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close()
        print("Grafica:", save_path)

    if "Vol_GT_mL" in df_res.columns and "Vol_Pred_mL" in df_res.columns:
        x = pd.to_numeric(df_res["Vol_GT_mL"], errors="coerce")
        y = pd.to_numeric(df_res["Vol_Pred_mL"], errors="coerce")
        valid = np.isfinite(x) & np.isfinite(y)
        x = x[valid]
        y = y[valid]

        if len(x) > 1:
            plt.figure(figsize=(8, 8))
            plt.scatter(x, y, alpha=0.7, edgecolors="k")

            min_val = min(x.min(), y.min())
            max_val = max(x.max(), y.max())
            plt.plot([min_val, max_val], [min_val, max_val], "k--", alpha=0.7, label="Ideal agreement")

            m, b = np.polyfit(x, y, 1)
            y_fit = m * x + b
            plt.plot(x, y_fit, linewidth=2, label=f"Linear fit: y = {m:.2f}x + {b:.2f}")

            corr_matrix = np.corrcoef(x, y)
            r_sq = corr_matrix[0, 1] ** 2

            plt.text(
                0.05,
                0.90,
                f"$R^2 = {r_sq:.3f}$",
                transform=plt.gca().transAxes,
                fontsize=12,
                bbox=dict(facecolor="white", alpha=0.7),
            )
            plt.title("Volume Agreement: Ground Truth vs Prediction", fontsize=13, fontweight="bold")
            plt.xlabel("Ground Truth Volume (mL)")
            plt.ylabel("Predicted Volume (mL)")
            plt.legend()
            plt.grid(True, linestyle="--", alpha=0.5)
            plt.tight_layout()

            save_path = os.path.join(out_dir, f"{prefix}_scatter_volume.png")
            plt.savefig(save_path, dpi=300, bbox_inches="tight")
            plt.close()
            print("Grafica:", save_path)

            mean_vol = (x + y) / 2.0
            diff_vol = y - x
            md = np.mean(diff_vol)
            sd = np.std(diff_vol)

            plt.figure(figsize=(10, 6))
            plt.scatter(mean_vol, diff_vol, alpha=0.7, edgecolors="k")
            plt.axhline(md, linestyle="-", linewidth=2, label=f"Mean difference = {md:.2f} mL")
            plt.axhline(md + 1.96 * sd, linestyle="--", linewidth=2, label=f"+1.96 SD = {md + 1.96 * sd:.2f} mL")
            plt.axhline(md - 1.96 * sd, linestyle="--", linewidth=2, label=f"-1.96 SD = {md - 1.96 * sd:.2f} mL")
            plt.title("Bland-Altman Plot for Tumor Volume", fontsize=13, fontweight="bold")
            plt.xlabel("Mean Volume (mL)")
            plt.ylabel("Prediction Error (Pred - GT) mL")
            plt.legend()
            plt.grid(True, linestyle="--", alpha=0.5)
            plt.tight_layout()

            save_path = os.path.join(out_dir, f"{prefix}_bland_altman.png")
            plt.savefig(save_path, dpi=300, bbox_inches="tight")
            plt.close()
            print("Grafica:", save_path)

    if "Dice" in df_res.columns:
        dice_values = pd.to_numeric(df_res["Dice"], errors="coerce").dropna()

        if len(dice_values) > 0:
            plt.figure(figsize=(8, 6))
            plt.hist(dice_values, bins=20, edgecolor="black", alpha=0.8)
            plt.axvline(dice_values.mean(), linestyle="--", linewidth=2, label=f"Mean = {dice_values.mean():.3f}")
            plt.title("Dice Score Distribution", fontsize=13, fontweight="bold")
            plt.xlabel("Dice Score")
            plt.ylabel("Number of Cases")
            plt.xlim(0, 1.0)
            plt.legend()
            plt.grid(axis="y", linestyle="--", alpha=0.5)
            plt.tight_layout()

            save_path = os.path.join(out_dir, f"{prefix}_histogram_dice.png")
            plt.savefig(save_path, dpi=300, bbox_inches="tight")
            plt.close()
            print("Grafica:", save_path)

    if "HD95" in df_res.columns:
        hd_values = pd.to_numeric(df_res["HD95"], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()

        if len(hd_values) > 0:
            plt.figure(figsize=(8, 6))
            plt.hist(hd_values, bins=20, edgecolor="black", alpha=0.8)
            plt.axvline(hd_values.mean(), linestyle="--", linewidth=2, label=f"Mean = {hd_values.mean():.2f} mm")
            plt.title("HD95 Distribution", fontsize=13, fontweight="bold")
            plt.xlabel("HD95 (mm)")
            plt.ylabel("Number of Cases")
            plt.legend()
            plt.grid(axis="y", linestyle="--", alpha=0.5)
            plt.tight_layout()

            save_path = os.path.join(out_dir, f"{prefix}_histogram_hd95.png")
            plt.savefig(save_path, dpi=300, bbox_inches="tight")
            plt.close()
            print("Grafica:", save_path)


def generate_fold_summary_plots(fold_summary_csv, out_dir):
    """Grafica metricas promedio por fold."""
    if not os.path.exists(fold_summary_csv):
        return

    df = pd.read_csv(fold_summary_csv)
    if df.empty or "Fold" not in df.columns:
        return

    if {"Dice_mean", "Dice_std"}.issubset(df.columns):
        plt.figure(figsize=(8, 5))
        plt.bar(df["Fold"].astype(str), df["Dice_mean"], yerr=df["Dice_std"], capsize=5)
        plt.ylim(0, 1.02)
        plt.title("Held-out Dice by Fold", fontsize=13, fontweight="bold")
        plt.xlabel("Fold")
        plt.ylabel("Dice mean +/- SD")
        plt.grid(axis="y", linestyle="--", alpha=0.5)
        plt.tight_layout()
        save_path = os.path.join(out_dir, "kfold_dice_by_fold.png")
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close()
        print("Grafica:", save_path)

    if {"HD95_mean", "HD95_std"}.issubset(df.columns):
        plt.figure(figsize=(8, 5))
        plt.bar(df["Fold"].astype(str), df["HD95_mean"], yerr=df["HD95_std"], capsize=5)
        plt.title("Held-out HD95 by Fold", fontsize=13, fontweight="bold")
        plt.xlabel("Fold")
        plt.ylabel("HD95 mean +/- SD (mm)")
        plt.grid(axis="y", linestyle="--", alpha=0.5)
        plt.tight_layout()
        save_path = os.path.join(out_dir, "kfold_hd95_by_fold.png")
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close()
        print("Grafica:", save_path)


def save_fold_summary(all_results_df, out_dir):
    """Guarda resumen por fold para reportar variabilidad."""
    aggregations = {
        "Case_ID": "count",
        "Dice": ["mean", "std", "median"],
        "IoU": ["mean", "std", "median"],
        "HD95": ["mean", "std", "median"],
        "ASD": ["mean", "std", "median"],
        "Vol_GT_mL": ["mean", "std"],
        "Vol_Pred_mL": ["mean", "std"],
    }

    available_aggregations = {
        column: agg for column, agg in aggregations.items() if column in all_results_df.columns
    }

    fold_summary = all_results_df.groupby("Fold").agg(available_aggregations)
    fold_summary.columns = [
        f"{metric}_{stat}" if stat != "count" else "n_cases"
        for metric, stat in fold_summary.columns
    ]
    fold_summary = fold_summary.reset_index()

    path = os.path.join(out_dir, "kfold_summary_by_fold.csv")
    fold_summary.to_csv(path, index=False)
    print("Resumen por fold:", path)
    return fold_summary, path


# ============================================================
# PIPELINE PRINCIPAL
# ============================================================

def run_kfold_pipeline():
    """
    Ejecuta entrenamiento + evaluacion + graficas.

    Protocolo:
    1. K-fold externo: cada fold held-out se reserva para metricas finales.
    2. Del train de cada fold se separa una validacion interna.
    3. El mejor checkpoint se elige SOLO con la validacion interna.
    4. Dice, IoU, HD95, ASD, volumen y graficas se reportan en los held-out folds.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Usando device:", device)

    datalist = build_datalist(DATA_ROOT)
    total_found = len(datalist)
    print("Total pacientes encontrados:", total_found)

    if total_found == 0:
        print("No se encontraron datos. Verifica DATA_ROOT.")
        return

    if PROBAR_CON_POCOS_DATOS:
        datalist = datalist[:CANTIDAD_PACIENTES]
        print(f"MODO PRUEBA: usando {len(datalist)} pacientes.")
    else:
        print(f"MODO COMPLETO: usando {len(datalist)} pacientes.")

    if len(datalist) < K_FOLDS:
        raise ValueError(f"K_FOLDS={K_FOLDS} es mayor que el numero de casos ({len(datalist)}).")

    train_transforms, eval_transforms = get_transforms()
    kfold = KFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

    all_fold_results = []
    all_training_logs = []
    split_manifest_rows = []
    fold_run_summary_rows = []

    for fold, (train_pool_idx, held_out_idx) in enumerate(kfold.split(datalist), start=1):
        if FOLDS_TO_RUN is not None and fold not in FOLDS_TO_RUN:
            continue

        set_determinism(seed=SEED + fold)

        fold_out_dir = os.path.join(OUT_DIR, f"fold_{fold:02d}")
        os.makedirs(fold_out_dir, exist_ok=True)

        train_pool_files = [datalist[idx] for idx in train_pool_idx]
        held_out_files = [datalist[idx] for idx in held_out_idx]
        train_files, inner_val_files = make_inner_split(train_pool_files, fold)

        for sample in train_files:
            split_manifest_rows.append({"Fold": fold, "Role": "train", "Case_ID": sample["case_id"]})
        for sample in inner_val_files:
            split_manifest_rows.append({"Fold": fold, "Role": "inner_val_checkpoint", "Case_ID": sample["case_id"]})
        for sample in held_out_files:
            split_manifest_rows.append({"Fold": fold, "Role": "held_out_evaluation", "Case_ID": sample["case_id"]})

        print("\n" + "#" * 70)
        print(f"Fold {fold}/{K_FOLDS}")
        print(f"Train pool externo: {len(train_pool_files)}")
        print(f"Train real: {len(train_files)}")
        print(f"Validacion interna/checkpoint: {len(inner_val_files)}")
        print(f"Held-out evaluacion final: {len(held_out_files)}")
        print("#" * 70)

        completed, marker = is_fold_completed(fold)
        force_rerun = fold in FORCE_RERUN_FOLDS

        if RESUME_COMPLETED_FOLDS and completed and not force_rerun:
            print(f"Fold {fold} ya estaba completado. Se salta entrenamiento/evaluacion.")
            fold_results_df, training_log_df, run_summary_row = load_completed_fold_outputs(fold, marker)
            all_fold_results.append(fold_results_df)
            all_training_logs.append(training_log_df)
            fold_run_summary_rows.append(run_summary_row)
            clean_cuda()
            continue

        if force_rerun:
            print(f"Fold {fold} esta en FORCE_RERUN_FOLDS. Se repetira aunque exista marcador.")

        best_model_path, log_path, best_inner_dice, best_epoch = train_one_fold(
            fold=fold,
            train_files=train_files,
            inner_val_files=inner_val_files,
            train_transforms=train_transforms,
            eval_transforms=eval_transforms,
            device=device,
        )

        generate_training_curves(log_path, fold_out_dir, fold_label=fold)

        fold_results_df, fold_results_csv = evaluate_fold_checkpoint(
            fold=fold,
            model_path=best_model_path,
            held_out_files=held_out_files,
            eval_transforms=eval_transforms,
            device=device,
        )

        save_numeric_summary(
            fold_results_df,
            fold_out_dir,
            f"heldout_summary_metrics_fold_{fold:02d}.csv",
        )
        generate_thesis_plots(fold_results_csv, fold_out_dir, prefix=f"fold_{fold:02d}")

        write_fold_completion_marker(
            fold=fold,
            train_pool_files=train_pool_files,
            train_files=train_files,
            inner_val_files=inner_val_files,
            held_out_files=held_out_files,
            best_model_path=best_model_path,
            log_path=log_path,
            fold_results_csv=fold_results_csv,
            best_inner_dice=best_inner_dice,
            best_epoch=best_epoch,
        )

        all_fold_results.append(fold_results_df)

        if os.path.exists(log_path):
            all_training_logs.append(pd.read_csv(log_path))

        fold_run_summary_rows.append(
            {
                "Fold": fold,
                "train_pool_n": len(train_pool_files),
                "train_n": len(train_files),
                "inner_val_n": len(inner_val_files),
                "held_out_n": len(held_out_files),
                "best_inner_val_dice": best_inner_dice,
                "best_epoch": best_epoch,
                "checkpoint": best_model_path,
                "resumed_from_completed_fold": False,
            }
        )

        clean_cuda()

    manifest_path = os.path.join(OUT_DIR, "kfold_split_manifest.csv")
    pd.DataFrame(split_manifest_rows).to_csv(manifest_path, index=False)
    print("\nManifest de splits:", manifest_path)

    run_summary_path = os.path.join(OUT_DIR, "kfold_run_summary.csv")
    pd.DataFrame(fold_run_summary_rows).to_csv(run_summary_path, index=False)
    print("Resumen de ejecucion:", run_summary_path)

    if all_training_logs:
        all_logs_df = pd.concat(all_training_logs, ignore_index=True)
        all_logs_path = os.path.join(OUT_DIR, "kfold_training_log_all_folds.csv")
        all_logs_df.to_csv(all_logs_path, index=False)
        print("Log combinado:", all_logs_path)
        generate_kfold_training_overview(all_logs_path, OUT_DIR)

    if all_fold_results:
        all_results_df = pd.concat(all_fold_results, ignore_index=True)
        all_results_path = os.path.join(OUT_DIR, "kfold_heldout_detailed_results_all_folds.csv")
        all_results_df.to_csv(all_results_path, index=False)
        print("Resultados held-out combinados:", all_results_path)

        save_numeric_summary(
            all_results_df,
            OUT_DIR,
            "kfold_heldout_summary_metrics_all_folds.csv",
        )

        _, fold_summary_path = save_fold_summary(all_results_df, OUT_DIR)
        generate_thesis_plots(all_results_path, OUT_DIR, prefix="kfold_heldout")
        generate_fold_summary_plots(fold_summary_path, OUT_DIR)

    print("\nPipeline k-fold terminado.")
    print("Carpeta de salida:", OUT_DIR)


if __name__ == "__main__":
    run_kfold_pipeline()
